# Séance de travail : nowcast du PIB de la zone euro

Ce notebook remplace le terminal. Chaque section fait l'équivalent d'une commande `eanowcast`,
affiche le résultat et **laisse une trace** : une ligne datée dans le journal, des fichiers dans un
dossier du jour.

### Mode d'emploi (VS Code)

1. **Ouvrir** : lance VS Code ; il rouvre le dossier `ea-gdpnow` de la dernière fois. Ouvre ce
   notebook depuis l'explorateur de fichiers (à gauche), dans `notebooks/`.
2. **Vérifier le noyau** en haut à droite du notebook : il doit indiquer `.venv`. Sinon, clique
   dessus → *Environnements Python* → `.venv`.
3. **Régler** la cellule *Paramètres de la séance* (section 0). C'est la seule cellule à modifier.
4. **Exécuter tout** : bouton *Exécuter tout* (*Run All*) en haut du notebook. Une cellule en cours
   affiche une icône qui tourne et un chronomètre à gauche.
5. **Lire** les résultats section par section. Chaque section commence par une explication.
6. **Fin de séance** : `Ctrl+S`, puis exécute la dernière cellule (`Maj+Entrée`) pour garder une
   copie HTML de la séance.

Pour relire le journal : ouvre `journal/journal.md` dans VS Code, puis `Ctrl+Maj+V` pour l'aperçu
mis en forme. Les CSV et les PNG du dossier du jour s'ouvrent aussi directement dans VS Code.

### Où sont les traces

| Trace | Emplacement | Contenu |
| --- | --- | --- |
| Journal | `journal/journal.md` | une ligne datée par action : téléchargement, nowcast, backtest, notes |
| Dossier du jour | `outputs/AAAA-MM-JJ/` | tableaux CSV, graphiques PNG, copie HTML de la séance |
| Base de données | `data/eanowcast.sqlite` | toutes les données téléchargées (par millésime) et les nowcasts enregistrés |

### Équivalences avec le terminal

| Commande terminal | Section du notebook |
| --- | --- |
| `eanowcast fetch` | 2. Télécharger les données |
| `eanowcast nowcast --save` | 4. Nowcast du jour |
| (pas d'équivalent) | 5. Qualité du modèle dans l'échantillon |
| `eanowcast track --target 2026Q3` | 6. Évolution du nowcast |
| `eanowcast history --target 2026Q3` | 7. Nowcasts enregistrés et bilan |
| `eanowcast backtest` | 8. Qualité hors échantillon : backtest |

### Quand lancer quoi

| Moment | Réglages |
| --- | --- |
| Séance normale (après des publications Eurostat) | `TELECHARGER = True`, `SAUVEGARDER = True` |
| Veille d'une estimation flash du PIB | idem : c'est le nowcast qui sera comparé au chiffre publié |
| Après la flash (le lendemain, après 11 h) | `TELECHARGER = True` : la section 7 calcule l'erreur |
| Après une modification du modèle (`series.toml` ou un `.py`) | lire la section 5, puis `LANCER_BACKTEST = True` |

Règle du projet : on explore dans un notebook, on modifie le modèle dans les fichiers `.py`.

## 0. Paramètres de la séance

In [ ]:
# ============ Seule cellule à modifier ============
TELECHARGER = True          # True : récupère les dernières données Eurostat (1 à 3 min)
CIBLE = None                # trimestre visé par le nowcast, ex. "2026Q4" ; None = trimestre en cours
TRIMESTRE_SUIVI = "2026Q4"  # trimestre dont on suit l'évolution (section 6) et fait le bilan (section 7)
SAUVEGARDER = True          # True : enregistre le nowcast du jour dans la base (trace datée)
LANCER_BACKTEST = False     # True : évaluation historique (quelques minutes)
DEBUT_BACKTEST = "2010Q1"   # premier trimestre évalué par le backtest
METHODE = "ar"              # prolongement des mois manquants : "ar" (rapide) ou "dfm" (facteur dynamique, lent)
NOTE = ""                   # remarque libre à garder dans le journal, ex. "testé un nouvel indicateur"

## 1. Préparation

Cette cellule charge le package, vérifie l'environnement, crée le dossier du jour et définit la
fonction `noter()` qui écrit dans le journal.

- Si un message **ATTENTION** parle de `.venv`, le notebook ne tourne pas dans l'environnement du
  projet. Clique sur le noyau en haut à droite → *Environnements Python* → `.venv`, puis relance
  tout.
- Si la base est absente, mets `TELECHARGER = True` : la section 2 la crée.

In [ ]:
import subprocess
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

import eanowcast
from eanowcast import benchmarks as bm
from eanowcast.backtest import backtest, error_by_component, metrics, track
from eanowcast.config import load_config
from eanowcast.data import download, load_from_store
from eanowcast.nowcast import run_nowcast
from eanowcast.report import tracking_chart
from eanowcast.store import Store

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)

DEBUT_SEANCE = time.time()
JOUR = datetime.now().strftime("%Y-%m-%d")
AUJOURDHUI = pd.Timestamp(JOUR)

# Le notebook est dans notebooks/ : la racine du repo est le dossier parent
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB = ROOT / "data" / "eanowcast.sqlite"
JOURNAL = ROOT / "journal" / "journal.md"
SORTIES = ROOT / "outputs" / JOUR
SORTIES.mkdir(parents=True, exist_ok=True)
NOTEBOOK = ROOT / "notebooks" / "02_seance_de_travail.ipynb"


def noter(action: str, texte: str) -> None:
    """Ajoute une ligne datée au journal (journal/journal.md) et l'affiche."""
    JOURNAL.parent.mkdir(parents=True, exist_ok=True)
    contenu = JOURNAL.read_text(encoding="utf-8") if JOURNAL.exists() else "# Journal du projet ea-gdpnow\n"
    if f"\n## {JOUR}\n" not in contenu:
        contenu += f"\n## {JOUR}\n\n"
    JOURNAL.write_text(contenu + f"- {datetime.now():%H:%M} · **{action}** · {texte}\n", encoding="utf-8")
    print(f"[journal] {action} · {texte}")


def nom_serie(sid: str) -> str:
    """'na:growth:P31_S14_S15' -> 'croissance consumption' ; 'ind:esi' -> 'esi'."""
    items = {cfg.na["gdp_item"]: "gdp", **{b.na_item: b.name for b in cfg.component_blocks}}
    parts = sid.split(":")
    if parts[0] == "na":
        return f"{'croissance' if parts[1] == 'growth' else 'niveau'} {items.get(parts[2], parts[2])}"
    return parts[-1]


cfg = load_config()
base_existait = DB.exists()
store = Store(DB)

print(f"Python    : {sys.version.split()[0]}  ->  {sys.executable}")
print(f"eanowcast : {eanowcast.__version__}   pandas {pd.__version__}")
print(f"Repo      : {ROOT}")
print(f"Base      : {DB}  ({'présente' if base_existait else 'absente : mets TELECHARGER = True'})")
print(f"Sorties   : {SORTIES}")
if ".venv" not in sys.executable:
    print("\nATTENTION : ce notebook ne tourne pas dans l'environnement .venv du projet. "
          "Clique sur le noyau en haut à droite -> Environnements Python -> .venv, puis relance tout.")

noter("ouverture", f"téléchargement={TELECHARGER}, cible={CIBLE or 'trimestre en cours'}, "
                   f"suivi={TRIMESTRE_SUIVI}, sauvegarde={SAUVEGARDER}, backtest={LANCER_BACKTEST}, "
                   f"méthode={METHODE}")

## 2. Télécharger les données Eurostat

Équivalent de `eanowcast fetch`. Le programme interroge l'API Eurostat pour les 22 séries
(12 séries de comptes nationaux, 10 indicateurs mensuels) et n'enregistre que les valeurs
**nouvelles ou révisées**. Chaque téléchargement est horodaté : la base garde l'historique des
révisions.

**Comment lire** : au premier téléchargement, toutes les valeurs sont nouvelles. Ensuite, 0 signifie
que rien n'a changé depuis la dernière fois ; un petit nombre signale une nouvelle publication (un
mois de plus) ou une révision. En cas d'erreur (réseau, limite de requêtes Eurostat), le notebook
continue avec la base existante ; attends 5 minutes et relance cette cellule.

In [ ]:
if TELECHARGER:
    debut = time.time()
    try:
        ecrits = download(cfg, store)
    except Exception as exc:
        noter("téléchargement", f"ÉCHEC ({type(exc).__name__} : {exc}) ; on continue avec la base existante")
    else:
        maj = {nom_serie(k): v for k, v in ecrits.items() if v > 0}
        texte = (f"{sum(ecrits.values())} valeurs nouvelles ou révisées dans {len(maj)} séries "
                 f"sur {len(ecrits)} ({time.time() - debut:.0f} s)")
        if 0 < len(maj) <= 8:
            texte += " : " + ", ".join(f"{k} ({v})" for k, v in maj.items())
        noter("téléchargement", texte)
else:
    print("Téléchargement non demandé (TELECHARGER = False) : on utilise la base existante.")

## 3. Charger et vérifier les données

On charge la dernière version de chaque série depuis la base, puis on vérifie la période couverte.

**Ce qu'il faut regarder** :
- `fin` : la dernière donnée publiée. Les enquêtes ont environ un mois d'avance sur la production
  industrielle, la construction et les ventes au détail ; les comptes nationaux s'arrêtent au dernier
  trimestre publié par composante.
- `nb valeurs` : une série beaucoup plus courte que les autres signale un problème de téléchargement.
- Le tableau des croissances : il doit correspondre aux chiffres publiés par Eurostat (aux arrondis près).

In [ ]:
ds = load_from_store(cfg, store)


def couverture(frame: pd.DataFrame, nature: str) -> pd.DataFrame:
    lignes = {}
    for col in frame.columns:
        s = frame[col].dropna()
        lignes[col] = {"nature": nature, "début": s.index.min(), "fin": s.index.max(), "nb valeurs": len(s)}
    return pd.DataFrame(lignes).T


cov = pd.concat([couverture(ds.growth, "trimestriel, croissance (%)"), couverture(ds.monthly, "mensuel")])
cov.to_csv(SORTIES / "couverture_donnees.csv")
display(cov)

display(Markdown("**Croissance en volume publiée (%, trimestre sur trimestre), 4 derniers trimestres**"))
display(ds.growth.dropna(how="all").tail(4).round(2))

pib = ds.growth["gdp"].dropna()
comp_fin = ds.growth.drop(columns="gdp").dropna().index.max()
fins = ds.monthly.apply(lambda s: s.last_valid_index())
mensuel = " ; ".join(f"jusqu'à {m} : {', '.join(fins[fins == m].index)}"
                     for m in sorted(fins.unique(), reverse=True))
noter("données", f"PIB publié jusqu'à {pib.index.max()} ({pib.iloc[-1]:+.2f} %), composantes jusqu'à "
                 f"{comp_fin} ; mensuelles {mensuel}")

## 4. Nowcast du jour

Équivalent de `eanowcast nowcast --save`. Le modèle utilise **toutes les données connues
aujourd'hui** et prévoit chaque trimestre entre le dernier trimestre publié par composante et la
cible. Par exemple, fin octobre 2026, il fait le *backcast* de 2026T3 (terminé mais pas encore
publié) et le *nowcast* de 2026T4.

**Comment lire chaque tableau** :
- `croissance (%)` : croissance prévue du bloc, trimestre sur trimestre ;
- `poids` : part nominale du bloc dans le PIB l'année précédente (les importations entrent avec un
  signe négatif dans la contribution) ;
- `contribution (pt)` : croissance × poids, en points de PIB. La ligne `gdp` est la somme des
  contributions : c'est le nowcast du PIB.

Le tableau **Comparaison** reprend la logique du graphique du GDPNow : la variation du nowcast
depuis le dernier nowcast enregistré se décompose exactement par bloc. Les **benchmarks** (AR(2) sur
le PIB, bridge direct sur le PIB, moyenne historique) servent de point de repère.

In [ ]:
cible = pd.Period(CIBLE, freq="Q") if CIBLE else pd.Period(AUJOURDHUI, freq="Q")
res = run_nowcast(ds, cfg, cible, fill_method=METHODE)
run_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")
print(f"Données du {JOUR} | dernier trimestre publié par composante : {res.last_component_quarter}")

noms_colonnes = {"label": "bloc", "growth": "croissance (%)", "weight": "poids",
                 "contribution": "contribution (pt)"}
resume = []
for q, table in res.by_quarter.items():
    flash = q in res.gdp_flash_used
    display(Markdown(f"### {q} " + ("(flash publiée : le PIB est calé dessus)" if flash else "")))
    display(table.rename(columns=noms_colonnes).round(3))

    # Comparaison avec le dernier nowcast enregistré pour ce trimestre
    precedents = store.read_nowcasts(str(q))
    if not precedents.empty:
        dernier = precedents[precedents["run_at"] == precedents["run_at"].max()].set_index("block")
        avant = dernier["contribution"].copy()
        avant["gdp"] = dernier.loc["gdp", "growth"]
        maintenant = table["contribution"].copy()
        maintenant["gdp"] = table.loc["gdp", "growth"]
        comp = pd.DataFrame({"précédent": avant.reindex(maintenant.index), "aujourd'hui": maintenant})
        comp["variation"] = comp["aujourd'hui"] - comp["précédent"]
        display(Markdown(f"**Comparaison avec le nowcast enregistré le {dernier['as_of'].iloc[0]}** "
                         "(contributions en points ; ligne gdp = croissance du PIB)"))
        display(comp.round(3))

    table.to_csv(SORTIES / f"nowcast_{q}.csv")
    if flash:  # PIB déjà publié : les benchmarks n'ont plus d'objet
        resume.append(f"{q} {table.loc['gdp', 'growth']:+.2f} % (calé sur la flash)")
        continue
    bench = {"AR(2)": bm.ar_gdp(ds.growth["gdp"], cfg, q),
             "bridge direct": bm.direct_bridge(ds.growth["gdp"], res.X, cfg, q),
             "moyenne": bm.historical_mean(ds.growth["gdp"], cfg)}
    print("Benchmarks : " + " | ".join(f"{k} {v:+.2f} %" for k, v in bench.items()))
    resume.append(f"{q} {table.loc['gdp', 'growth']:+.2f} % "
                  f"[AR(2) {bench['AR(2)']:+.2f}, bridge direct {bench['bridge direct']:+.2f}]")

if SAUVEGARDER:
    for q, table in res.by_quarter.items():
        store.write_nowcast(run_at, JOUR, str(q), table)
noter("nowcast", " ; ".join(resume) + (" | enregistré en base" if SAUVEGARDER else " | non enregistré"))

## 5. Qualité du modèle (dans l'échantillon)

Cette section examine les équations **estimées aujourd'hui** (celles qui ont produit le nowcast de la
section 4, stockées dans `res.models`). Elle répond à trois questions : chaque équation explique-t-elle
son bloc ? ses coefficients sont-ils significatifs et du bon signe ? ses résidus respectent-ils les
hypothèses des MCO ? Une quatrième table vérifie l'agrégation.

**Comment lire les tableaux** :

| Critère | Lecture | Que faire si c'est mauvais |
| --- | --- | --- |
| `R²`, `R² ajusté` | part de la variance du bloc expliquée par l'équation | un R² faible n'est pas rédhibitoire : seul le backtest (section 8) tranche |
| `σ résidus` | erreur type de l'équation, en points de croissance du bloc ; × poids = erreur en points de PIB | comparer les blocs entre eux |
| `p-value (HAC)` | significativité du coefficient, robuste à l'autocorrélation et à l'hétéroscédasticité (Newey-West, 4 retards) | p > 0,10 et mauvais signe : candidat au retrait dans `series.toml` |
| `signe conforme` | le coefficient a-t-il le signe attendu par l'économie ? | un mauvais signe signale souvent de la colinéarité |
| Ljung-Box (4 retards) | p < 0,05 : résidus autocorrélés, dynamique manquante | ajouter un retard (`ar_lag = true` dans le bloc) |
| Jarque-Bera | p < 0,05 : résidus non normaux (valeurs extrêmes) | touche les intervalles, pas la prévision ponctuelle |
| Breusch-Pagan | p < 0,05 : hétéroscédasticité | lire les p-values HAC plutôt que MCO (c'est déjà le cas) |
| CUSUM (résidus MCO) | p < 0,05 : coefficients instables dans le temps | fenêtre d'estimation plus courte, ou rupture à traiter |

Avec 6 équations et 4 tests, environ une alerte sur 24 apparaît par pur hasard au seuil de 5 % :
on cherche des alertes **répétées** ou **cohérentes** (par exemple autocorrélation et instabilité sur
le même bloc), pas une alerte isolée.

**Agrégation** : l'écart-type de la contribution résiduelle (stocks + écart de chaînage) dit quelle part
de la volatilité du PIB échappe aux cinq composantes. L'ajustement agrégé reconstruit le PIB à partir
des équations, avec les indicateurs observés et, comme dans le nowcast, des importations calculées à
partir des composantes ajustées : une erreur sur les exportations se répercute en partie sur les
importations et se compense dans le PIB. Comparé au RMSFE du backtest à J+25 (section 8), l'écart
mesure le coût de l'information manquante en temps réel et de l'instabilité hors échantillon.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox, breaks_cusumolsresid, het_breuschpagan
from statsmodels.stats.stattools import jarque_bera
from eanowcast.transforms import historical_contributions, nominal_weights

# Signe attendu de chaque régresseur (vide = pas d'a priori clair)
SIGNES_ATTENDUS = {
    "consumption": {"retail_volume": "+", "consumer_conf": "+", "retail_conf": "+"},
    "government": {"lag1": "+"},
    "investment": {"construction": "+", "ip_capital_goods": "+", "construction_conf": "+"},
    "exports": {"ip_total": "+", "export_orders": "+"},
    "imports": {"block:consumption": "+", "block:investment": "+", "block:exports": "+"},
}
SEUIL = 0.05

ajust, coefs, diags, alertes = [], [], [], {}
for nom, r in res.models.items():
    X, noms = r.model.exog, r.model.exog_names
    e = np.asarray(r.resid)
    hac = sm.OLS(r.model.endog, X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
    ajust.append({"bloc": nom, "n": int(r.nobs), "R²": r.rsquared, "R² ajusté": r.rsquared_adj,
                  "σ résidus (pt)": float(np.sqrt(r.scale)), "p-value F": r.f_pvalue})
    for j, var in enumerate(noms):
        attendu = SIGNES_ATTENDUS.get(nom, {}).get(var, "")
        conforme = "" if not attendu else ("oui" if np.sign(hac.params[j]) == (1 if attendu == "+" else -1) else "NON")
        coefs.append({"bloc": nom, "variable": var, "coef": hac.params[j], "t (HAC)": hac.tvalues[j],
                      "p-value (HAC)": hac.pvalues[j], "signe attendu": attendu, "signe conforme": conforme})
    tests = {"Ljung-Box": acorr_ljungbox(e, lags=[4], return_df=True)["lb_pvalue"].iloc[0],
             "Jarque-Bera": jarque_bera(e)[1],
             "Breusch-Pagan": het_breuschpagan(e, X)[1],
             "CUSUM": breaks_cusumolsresid(e, ddof=X.shape[1])[1]}
    noms_alertes = {"Ljung-Box": "autocorrélation", "Jarque-Bera": "non-normalité",
                    "Breusch-Pagan": "hétéroscédasticité", "CUSUM": "instabilité"}
    a = [noms_alertes[t] for t, p in tests.items() if p < SEUIL]
    a += [f"signe de {c['variable']}" for c in coefs if c["bloc"] == nom and c["signe conforme"] == "NON"]
    alertes[nom] = a
    diags.append({"bloc": nom, **{f"p {t}": p for t, p in tests.items()}, "alertes (5 %)": ", ".join(a) or "aucune"})

ajust = pd.DataFrame(ajust).set_index("bloc")
coefs = pd.DataFrame(coefs).set_index(["bloc", "variable"])
diags = pd.DataFrame(diags).set_index("bloc")

# Agrégation : contributions historiques et ajustement agrégé du PIB dans l'échantillon
names = [b.name for b in cfg.component_blocks]
signes = pd.Series({b.name: b.sign for b in cfg.component_blocks})
hc = historical_contributions(ds.growth, ds.levels, cfg)
hc = hc[~hc.index.isin(cfg.exclude_quarters)]
pib_hist = ds.growth["gdp"].reindex(hc.index)
agreg = pd.DataFrame({"moyenne (pt)": hc.mean(), "écart-type (pt)": hc.std(), "corrélation avec le PIB": hc.corrwith(pib_hist)})
agreg.loc["PIB (croissance)"] = [pib_hist.mean(), pib_hist.std(), 1.0]

# Valeurs ajustées "cohérentes avec le nowcast" : comme en prévision, les importations reçoivent les
# composantes AJUSTÉES (et non observées), ce qui reproduit la compensation exportations/importations.
ajuste = {}
for nom, r in res.models.items():  # même ordre que series.toml
    Xd = pd.DataFrame(r.model.exog, index=r.model.data.row_labels, columns=r.model.exog_names)
    for col in Xd.columns:
        if col.startswith("block:"):
            Xd[col] = ajuste[col.split(":", 1)[1]].reindex(Xd.index)
    ajuste[nom] = Xd.dot(r.params).dropna()
communs = sorted(set.intersection(*(set(v.index) for v in ajuste.values())))
pib_ajuste = pd.Series({q: float((signes * nominal_weights(ds.levels, q, names) * pd.Series({b: ajuste[b][q] for b in names})).sum()
                                 + ajuste["residual"][q]) for q in communs})
pib_obs = ds.growth["gdp"].reindex(pib_ajuste.index)
rmse_in = float(np.sqrt(((pib_ajuste - pib_obs) ** 2).mean()))
r2_agrege = 1 - ((pib_ajuste - pib_obs) ** 2).sum() / ((pib_obs - pib_obs.mean()) ** 2).sum()
somme_poids = float((signes * nominal_weights(ds.levels, cible, names)).sum())

display(Markdown("**Ajustement par bloc**"))
display(ajust.round(3))
display(Markdown("**Coefficients** (p-values robustes HAC, Newey-West 4 retards)"))
display(coefs.round(3))
display(Markdown("**Tests sur les résidus** (p-values ; alerte si < 5 %)"))
display(diags.round(3))
display(Markdown("**Agrégation : contributions historiques** (points de PIB, hors 2020T1–2021T2)"))
display(agreg.round(3))
print(f"Ajustement agrégé du PIB dans l'échantillon ({len(communs)} trimestres, indicateurs observés) : "
      f"R² = {r2_agrege:.2f}, RMSE = {rmse_in:.2f} pt  (à comparer au RMSFE du backtest à J+25, section 8)")
print(f"Somme des poids signés des 5 composantes ({cible.year - 1}) : {somme_poids:.3f} "
      f"-> stocks et objets de valeur ≈ {100 * (1 - somme_poids):.1f} % du PIB")

ajust.to_csv(SORTIES / "qualite_ajustement.csv")
coefs.to_csv(SORTIES / "qualite_coefficients.csv")
diags.to_csv(SORTIES / "qualite_tests_residus.csv")
agreg.to_csv(SORTIES / "qualite_agregation.csv")
avec_alerte = {k: v for k, v in alertes.items() if v}
noter("qualité", "R² : " + ", ".join(f"{b} {v:.2f}" for b, v in ajust["R²"].items())
      + f" ; PIB agrégé R² {r2_agrege:.2f}, RMSE {rmse_in:.2f} pt ; alertes : "
      + ("; ".join(f"{b} ({', '.join(v)})" for b, v in avec_alerte.items()) or "aucune"))

## 6. Évolution du nowcast d'un trimestre (pseudo temps réel)

Équivalent de `eanowcast track`. Pour chaque vendredi, du mois précédant le trimestre suivi
jusqu'à 25 jours après sa fin, le modèle refait le nowcast **en masquant les données qui
n'étaient pas encore publiées** à cette date (d'après les délais de publication de `series.toml`).

**Comment lire le graphique** : la ligne noire est le nowcast du PIB, les barres empilées sont les
contributions des blocs. Quand une barre change d'une semaine à l'autre, c'est ce bloc qui a fait
bouger le nowcast. Si le PIB est déjà publié, une ligne pointillée le montre.

**Limite** : on utilise la dernière version des données, donc les révisions ultérieures sont
ignorées. La section 7 montre au contraire les nowcasts réellement enregistrés au fil des séances.

In [ ]:
q_suivi = pd.Period(TRIMESTRE_SUIVI, freq="Q")
debut = (q_suivi.start_time - pd.Timedelta(days=30)).normalize()
fin = min(q_suivi.end_time.normalize() + pd.Timedelta(days=25), AUJOURDHUI)
dates = list(pd.date_range(debut, fin, freq="W-FRI"))
reel = ds.growth["gdp"].get(q_suivi, np.nan)

if not dates:
    print(f"Aucun vendredi entre le {debut:%d/%m/%Y} et aujourd'hui : rien à suivre pour {q_suivi} pour l'instant.")
else:
    tr = track(ds, cfg, str(q_suivi), dates, fill_method=METHODE)
    png = tracking_chart(tr, cfg, str(q_suivi), SORTIES / f"suivi_{q_suivi}.png",
                         actual=None if pd.isna(reel) else float(reel))
    tr.to_csv(SORTIES / f"suivi_{q_suivi}.csv")
    display(Image(filename=str(png)))
    display(tr.round(2))
    noter("suivi", f"{q_suivi} en pseudo temps réel : {tr['gdp'].iloc[0]:+.2f} % au {dates[0]:%d/%m} -> "
                   f"{tr['gdp'].iloc[-1]:+.2f} % au {dates[-1]:%d/%m} ({len(dates)} dates)")

## 7. Nowcasts enregistrés et bilan

Équivalent de `eanowcast history`. Contrairement à la section 6, ce sont les nowcasts **réellement
produits** lors de tes séances (avec `SAUVEGARDER = True`), avec les données disponibles ce jour-là.
C'est la trace honnête de ce que le modèle disait avant la publication.

**Bilan** : dès que le PIB du trimestre suivi est publié (estimation flash à environ 30 jours), la
cellule compare le dernier nowcast enregistré **avant** la publication au chiffre publié.

In [ ]:
hist_nc = store.read_nowcasts(str(q_suivi))
if hist_nc.empty:
    print(f"Aucun nowcast enregistré pour {q_suivi}. Les traces commencent dès qu'une séance tourne "
          "avec SAUVEGARDER = True.")
else:
    derniers = hist_nc.sort_values("run_at").groupby("as_of")["run_at"].max()  # dernier run de chaque jour
    h = hist_nc[hist_nc["run_at"].isin(derniers)]
    large = h.pivot_table(index="as_of", columns="block", values="contribution")
    large["gdp"] = h[h["block"] == "gdp"].set_index("as_of")["growth"]
    large.index = pd.to_datetime(large.index)
    display(Markdown(f"**Nowcasts de {q_suivi} enregistrés (dernier run de chaque jour)**"))
    display(large.round(3))
    png = tracking_chart(large, cfg, str(q_suivi), SORTIES / f"historique_{q_suivi}.png",
                         actual=None if pd.isna(reel) else float(reel),
                         title=f"Nowcasts enregistrés en temps réel, {q_suivi}")
    display(Image(filename=str(png)))

# Bilan après publication
if pd.isna(reel):
    print(f"\nLe PIB de {q_suivi} n'est pas encore publié : le bilan sera possible après l'estimation flash.")
else:
    sid = f"na:growth:{cfg.na['gdp_item']}"
    publie_le = store.con.execute("SELECT MIN(retrieved_at) FROM observations WHERE series_id = ? AND period = ?",
                                  (sid, str(q_suivi))).fetchone()[0]
    avant_pub = hist_nc[(hist_nc["block"] == "gdp") & (hist_nc["run_at"] < publie_le)] if not hist_nc.empty else hist_nc
    if avant_pub.empty:
        print(f"\nPIB {q_suivi} publié : {reel:+.2f} %. Aucun nowcast n'avait été enregistré avant sa publication.")
    else:
        d = avant_pub.sort_values("run_at").iloc[-1]
        texte = (f"{q_suivi} : publié {reel:+.2f} % ; dernier nowcast enregistré avant publication "
                 f"({d['as_of']}) {d['growth']:+.2f} % ; erreur {d['growth'] - reel:+.2f} pt")
        deja = JOURNAL.exists() and f"**bilan** · {q_suivi} :" in JOURNAL.read_text(encoding="utf-8")
        if deja:
            print(texte + "  (déjà dans le journal)")
        else:
            noter("bilan", texte)

## 8. Qualité hors échantillon : backtest

Équivalent de `eanowcast backtest`. Pour chaque trimestre depuis `DEBUT_BACKTEST`, le modèle
refait le nowcast à 4 dates (60 et 30 jours avant la fin du trimestre, à la fin, 25 jours après),
avec uniquement les données publiées à ces dates, puis compare au PIB publié. Les trimestres
2020T1–2021T2 (Covid) sont exclus des scores : une seule erreur d'environ 10 points en 2020T2
écraserait tout le reste. Le fichier `backtest.csv` les garde quand même.

**Comment lire le tableau des scores** :
- `RMSFE` et `MAE` : erreurs moyennes en points de croissance (plus c'est bas, mieux c'est) ;
- `RMSFE / AR(2)` : < 1 = mieux que l'AR(2) sur le PIB ; on attend une baisse de J-60 à J+25 ;
- `DM vs components` : test de Diebold-Mariano ; **positif** = le modèle par composantes fait mieux
  que ce benchmark ; `p-value` < 0,10 = différence significative à 10 %.

- **Biais** : erreur moyenne (prévision − publié). |t| > 2 : le modèle surestime (biais > 0) ou
  sous-estime (biais < 0) systématiquement ; `erreur max` montre le pire trimestre.

Le dernier tableau donne l'erreur de contribution par bloc (en points de PIB) : il montre où le
modèle se trompe (cf. Higgins 2014, tableau 6). C'est le critère de qualité qui compte le plus pour
un nowcast : la section 5 aide à comprendre les équations, celle-ci dit si elles prévoient bien.

In [ ]:
if not LANCER_BACKTEST:
    print("Backtest non lancé (LANCER_BACKTEST = False).")
else:
    debut = time.time()
    fin_bt = str(ds.growth["gdp"].dropna().index.max())
    print(f"Backtest {DEBUT_BACKTEST} -> {fin_bt}, méthode {METHODE} : patiente quelques minutes...")
    bt = backtest(ds, cfg, DEBUT_BACKTEST, fin_bt, fill_method=METHODE, verbose=False)
    exclus = {str(q) for q in cfg.exclude_quarters}
    notes = bt[~bt["target"].isin(exclus)]
    m = metrics(notes)
    ec = error_by_component(notes, cfg)
    dossier = SORTIES / "backtest"
    dossier.mkdir(exist_ok=True)
    bt.to_csv(dossier / "backtest.csv", index=False)
    m.to_csv(dossier / "metrics.csv", index=False)
    ec.to_csv(dossier / "error_by_component.csv")

    display(Markdown(f"**Scores** ({notes['target'].nunique()} trimestres, hors 2020T1–2021T2)"))
    display(m.round(3))
    g = notes.groupby(["horizon", "model"], sort=False)["error"]
    biais = pd.DataFrame({"biais (pt)": g.mean(), "t": g.mean() / (g.std() / np.sqrt(g.count())),
                          "erreur max (pt)": g.apply(lambda e: e.abs().max())})
    biais.to_csv(dossier / "biais.csv")
    display(Markdown("**Biais et erreur maximale**"))
    display(biais.round(3))
    display(Markdown("**Erreur de contribution par bloc** (RMSE, points de PIB)"))
    display(ec.round(3))

    comp = m[m["model"] == "components"].set_index("horizon")["RMSFE / AR(2)"]
    direct = m[m["model"] == "Direct GDP bridge"].set_index("horizon")
    noter("backtest", f"{DEBUT_BACKTEST}-{fin_bt}, {notes['target'].nunique()} trimestres notés, méthode {METHODE} ; "
                      "RMSFE/AR(2) composantes : " + " · ".join(f"{h} {v:.2f}" for h, v in comp.items())
                      + f" ; bridge direct J0 : {direct.loc['J0', 'RMSFE / AR(2)']:.2f} "
                      f"(DM p = {direct.loc['J0', 'p-value']:.2f}) ; {time.time() - debut:.0f} s")

## 9. Note libre et résumé du journal

Si `NOTE` n'est pas vide, elle est ajoutée au journal. La cellule affiche ensuite les dernières
lignes du journal : c'est le résumé de la séance.

In [ ]:
if NOTE.strip():
    noter("note", NOTE.strip())
display(Markdown("\n".join(JOURNAL.read_text(encoding="utf-8").splitlines()[-15:])))
FIN_CALCULS = time.time()

## 10. Copie HTML de la séance

Pour garder le notebook **avec ses résultats** dans le dossier du jour :
1. enregistre le notebook (`Ctrl+S`) ;
2. clique dans la cellule ci-dessous et exécute-la (`Maj+Entrée`).

Lors d'un *Exécuter tout*, cette cellule affiche simplement le rappel ci-dessus : c'est normal.

In [ ]:
fin_calculs = globals().get("FIN_CALCULS", DEBUT_SEANCE)
if not NOTEBOOK.exists():
    print(f"Notebook introuvable : {NOTEBOOK} (a-t-il été renommé ?)")
elif NOTEBOOK.stat().st_mtime < fin_calculs:
    print("Enregistre d'abord le notebook (Ctrl+S), puis relance cette cellule (Maj+Entrée).")
else:
    nom = f"seance_{JOUR}_{datetime.now():%Hh%M}"
    r = subprocess.run([sys.executable, "-m", "nbconvert", "--to", "html", str(NOTEBOOK),
                        "--output", nom, "--output-dir", str(SORTIES)], capture_output=True, text=True)
    if r.returncode == 0:
        noter("copie HTML", f"outputs/{JOUR}/{nom}.html")
    else:
        print(r.stderr[-2000:])